# TrustSkin AI — Google Colab GPU Setup & Controlled Experiments Runner

**CS320 Fall 2026 — Project P08: Reliable Recognition of Difficult Skin Lesions**

This notebook runs the full controlled experimental suite on a Google Colab GPU:
1. Mount Google Drive and link/unzip `derm` into `data/derm/`.
2. Install `trustskin` in editable mode (`pip install -e .`) and verify unit tests.
3. **Exp 0 (Baseline):** CS320 3-block scratch CNN (`configs/baseline_cnn.yaml`).
4. **Exp 1 (Ablation A):** Regularized `UncertaintyCNN` + Augmentation + Class-Weighted CE (`configs/ablation_weighted_aug.yaml`).
5. **Exp 2 & 3 (Main Proposed System):** MC Dropout Uncertainty + Post-Hoc Temperature Scaling + Selective Risk (`configs/improved_mc_cnn.yaml`).
6. **Exp 4 (Comparative Reference):** Pretrained `ResNet18MC` + MC Dropout + Temperature Scaling (`configs/resnet18_mc.yaml`).
7. **Exp 5:** Aggregate comparison table (`outputs/tables/experiment_comparison.csv`).

In [ ]:
# 1. Verify GPU runtime
!nvidia-smi

In [ ]:
# 2. Mount Google Drive (store derm.zip and sync checkpoints/outputs here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone repository and install package
# Replace <your-repo-url> with your GitHub URL
!git clone <your-repo-url> /content/TrustSkinAI
%cd /content/TrustSkinAI
!pip install -q medimeta PyYAML
!pip install -q -e .

In [ ]:
# 4. Fast dataset staging onto Colab local SSD
# Option A (fastest): Upload derm.zip to Drive and unzip into /content/TrustSkinAI/data/derm
# !unzip -q /content/drive/MyDrive/TrustSkinAI/derm.zip -d /content/TrustSkinAI/data/
# Option B: Symlink already-extracted folder from Google Drive
# !ln -s /content/drive/MyDrive/TrustSkinAI/data/derm /content/TrustSkinAI/data/derm
!ls -la data/derm

In [ ]:
# 5. Run unit & dataset smoke tests
!pytest tests/ -v

In [ ]:
# 6. Exp 0: Train & Evaluate CS320 Reference Baseline CNN
!python scripts/02_train_baseline.py --config configs/baseline_cnn.yaml

In [ ]:
# 7. Exp 1 (Ablation A): Train Regularized UncertaintyCNN + Augmentation + Class Weights
!python scripts/03_train_improved.py --config configs/improved_mc_cnn.yaml

In [ ]:
# 8. Exp 2 & 3: Fit Temperature Scaling (Val) + MC Dropout + Selective Classification (Test)
!python scripts/04_calibrate_and_select.py --config configs/improved_mc_cnn.yaml --checkpoint checkpoints/improved_mc_cnn_best.pt

In [ ]:
# 9. Exp 4 (Comparative Reference): Pretrained ResNet-18 + MC Dropout + Temp Scaling
!python scripts/03_train_improved.py --config configs/resnet18_mc.yaml
!python scripts/04_calibrate_and_select.py --config configs/resnet18_mc.yaml --checkpoint checkpoints/resnet18_mc_best.pt

In [ ]:
# 10. Exp 5: Compile Comparison Table & Copy Artifacts to Google Drive
!python scripts/05_compare_experiments.py --output-dir outputs
# !mkdir -p /content/drive/MyDrive/TrustSkinAI/results
# !cp -r checkpoints outputs /content/drive/MyDrive/TrustSkinAI/results/